In [4]:
%pip install executorch

  Using cached executorch-1.1.0-cp313-cp313-macosx_12_0_arm64.whl.metadata (4.3 kB)
  Using cached expecttest-0.3.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached hypothesis-6.151.6-py3-none-any.whl.metadata (5.6 kB)
  Using cached kgb-7.3-py2.py3-none-any.whl.metadata (21 kB)
  Using cached pandas-3.0.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (79 kB)
  Using cached parameterized-0.9.0-py2.py3-none-any.whl.metadata (18 kB)
  Using cached pytest-8.4.2-py3-none-any.whl.metadata (7.7 kB)
  Using cached pytest_xdist-3.8.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached pytest_rerunfailures-15.1-py3-none-any.whl.metadata (20 kB)
  Using cached pytest_json_report-1.5.0-py3-none-any.whl.metadata (15 kB)
  Using cached pytorch_tokenizers-1.1.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached coremltools-9.0-cp313-none-macosx_11_0_arm64.whl.metadata (2.7 kB)
  Using cached scikit_learn-1.7.1-cp313-cp313-ma

In [3]:
import torch
import torch.nn as nn
from transformers import AutoModelForCausalLM
from transformers.models.lfm2.modeling_lfm2 import Lfm2ShortConv
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower

# --- 1. Define the Helper and Patch (Same as before) ---
def apply_mask_to_padding_states(hidden_states, attention_mask):
    if attention_mask is not None and attention_mask.shape[1] > 1 and attention_mask.shape[0] > 1:
        dtype = hidden_states.dtype
        hidden_states = (hidden_states * attention_mask[:, :, None]).to(dtype)
    return hidden_states

def patched_slow_forward(
    self,
    x: torch.Tensor,
    past_key_values = None,
    cache_position: torch.LongTensor | None = None,
    attention_mask: torch.Tensor | None = None,
):
    seqlen = x.shape[1]
    x = apply_mask_to_padding_states(x, attention_mask)
    BCx = self.in_proj(x).transpose(-1, -2)
    B, C, x = BCx.chunk(3, dim=-2)
    Bx = B * x

    # --- CHANGED: Force "Else" path (Prefill logic) ---
    if past_key_values is not None:
        conv_state = nn.functional.pad(Bx, (self.L_cache - Bx.shape[-1], 0))
        past_key_values.conv_cache[self.layer_idx].copy_(conv_state)

    conv_out = self.conv(Bx)[..., :seqlen]
    y = C * conv_out
    y = y.transpose(-1, -2).contiguous()
    y = self.out_proj(y)
    return y

# --- 2. Define a Wrapper to Sanitize Output ---
class ExportWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids):
        # We assume 'use_cache=False' or we simply ignore the returned cache object.
        # This forces the output to be simple Tensors, which Executorch accepts.
        outputs = self.model(input_ids=input_ids, use_cache=False)
        
        # Return ONLY the logits tuple. 
        # We strip out 'past_key_values' (Lfm2HybridConvCache) which causes the crash.
        return (outputs.logits,)

# --- 3. Main Execution ---
print("Applying monkey patch...")
original_slow_forward = Lfm2ShortConv.slow_forward
Lfm2ShortConv.slow_forward = patched_slow_forward

try:
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        "LiquidAI/LFM2.5-1.2B-Instruct", 
        trust_remote_code=True,
        torch_dtype=torch.float32 
    )
    model.eval()

    # Wrap the model so export sees a clean output signature
    wrapper = ExportWrapper(model)

    print("Preparing inputs...")
    # Use Integers for input_ids
    sample_inputs = (
        torch.randint(0, model.config.vocab_size, (1, 128), dtype=torch.long), 
    )

    print("Starting export (this captures the graph)...")
    # Export the WRAPPER, not the raw model
    exported_program = torch.export.export(wrapper, sample_inputs)
    
    print("Lowering to ExecuTorch (XNNPACK)...")
    et_program = to_edge_transform_and_lower(
        exported_program,
        partitioner=[XnnpackPartitioner()]
    ).to_executorch()

    output_filename = "LFM2.5-1.2B-Instruct.pte"
    with open(output_filename, "wb") as f:
        f.write(et_program.buffer)

    print(f"✅ Success! Exported to {output_filename}")

finally:
    Lfm2ShortConv.slow_forward = original_slow_forward
    print("Restored original model methods.")

Applying monkey patch...
Loading model...


Loading weights: 100%|█| 148/148 [00:01<00:00, 86.35it/s, Materializing param=mo


Preparing inputs...
Starting export (this captures the graph)...
Lowering to ExecuTorch (XNNPACK)...


/opt/miniconda3/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/opt/miniconda3/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/opt/miniconda3/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/opt/miniconda3/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/opt/miniconda3/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespe

✅ Success! Exported to LFM2.5-1.2B-Instruct.pte
Restored original model methods.


In [5]:
import torch
# Use the core Runtime API instead of the extension
from executorch.runtime import Runtime 

# 1. Setup Input (Must match the exported shape!)
sequence_length = 128
vocab_size = model.config.vocab_size
input_ids = torch.randint(0, vocab_size, (1, sequence_length), dtype=torch.long)

print("Loading ExecuTorch runtime...")
runtime = Runtime.get()

print(f"Loading program: LFM2.5-1.2B-Instruct.pte")
program = runtime.load_program("LFM2.5-1.2B-Instruct.pte")

# Load the 'forward' method (this corresponds to the forward() function we exported)
method = program.load_method("forward")

print("Running ExecuTorch inference...")
# execute() takes a list of tensors and returns a list of tensors
et_outputs = method.execute([input_ids])
et_logits = et_outputs[0]

print("✅ ExecuTorch run successful.")

# --- Verification (Same as before) ---

print("Running PyTorch Eager reference...")
with torch.no_grad():
    # Make sure 'wrapper' is defined from the previous step
    pt_outputs = wrapper(input_ids)
    pt_logits = pt_outputs[0]

print("\n--- Comparison ---")
# Check standard deviation of the difference
diff = (et_logits - pt_logits).abs()
max_diff = diff.max().item()
mean_diff = diff.mean().item()

print(f"Max Difference: {max_diff:.6f}")
print(f"Mean Difference: {mean_diff:.6f}")

is_match = torch.allclose(et_logits, pt_logits, atol=1e-3, rtol=1e-3)

if is_match:
    print("🎉 SUCCESS: ExecuTorch output matches PyTorch!")
else:
    print("⚠️ WARNING: Outputs differ slightly (likely due to float precision/XNNPACK optimizations).")

[program.cpp:154] InternalConsistency verification requested but not available


Loading ExecuTorch runtime...
Loading program: LFM2.5-1.2B-Instruct.pte


[cpuinfo_utils.cpp:71] Reading file /sys/devices/soc0/image_version
[cpuinfo_utils.cpp:87] Failed to open midr file /sys/devices/soc0/image_version


Running ExecuTorch inference...
✅ ExecuTorch run successful.
Running PyTorch Eager reference...

--- Comparison ---
Max Difference: 0.000261
Mean Difference: 0.000020
🎉 SUCCESS: ExecuTorch output matches PyTorch!


In [6]:
import torch
from transformers import AutoTokenizer
from executorch.runtime import Runtime

# --- Configuration ---
MODEL_PATH = "LFM2.5-1.2B-Instruct.pte"
# This must match the length you used in torch.randint during export!
STATIC_SEQUENCE_LENGTH = 128 

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("LiquidAI/LFM2.5-1.2B-Instruct")
# Ensure we have a pad token (some models don't have one by default)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

print(f"Loading ExecuTorch Program: {MODEL_PATH}...")
runtime = Runtime.get()
program = runtime.load_program(MODEL_PATH)
method = program.load_method("forward")

def generate_text(prompt, max_new_tokens=20):
    print(f"\n🔵 Input: '{prompt}'")
    print("🟢 Generating: ", end="", flush=True)
    
    generated_text = prompt
    
    for _ in range(max_new_tokens):
        # 1. Tokenize the current text
        input_ids = tokenizer.encode(generated_text, return_tensors="pt")
        
        # 2. Get the index of the last real token (where we want the prediction)
        # e.g., if we have 5 tokens, the prediction is at index 4 (0-based)
        last_real_token_idx = input_ids.shape[1] - 1
        
        # 3. Handle Static Shape (Padding / Truncating)
        # We MUST provide exactly STATIC_SEQUENCE_LENGTH tokens to the model.
        if input_ids.shape[1] > STATIC_SEQUENCE_LENGTH:
            # If context is too long, keep the last 128 tokens
            input_ids = input_ids[:, -STATIC_SEQUENCE_LENGTH:]
            last_real_token_idx = STATIC_SEQUENCE_LENGTH - 1
        elif input_ids.shape[1] < STATIC_SEQUENCE_LENGTH:
            # If context is too short, PAD to the right
            pad_len = STATIC_SEQUENCE_LENGTH - input_ids.shape[1]
            padding = torch.full((1, pad_len), tokenizer.pad_token_id, dtype=torch.long)
            input_ids = torch.cat([input_ids, padding], dim=1)
            
        # 4. Run Inference
        # ExecuTorch expects inputs as a list of tensors
        outputs = method.execute([input_ids])
        
        # 5. Decode Output
        # logits shape: [1, 128, vocab_size]
        logits = outputs[0]
        
        # We only care about the logits for the last REAL token we input.
        # (The model predicts the NEXT token at that position).
        next_token_logits = logits[0, last_real_token_idx, :]
        
        # Greedy decoding (pick the highest probability)
        next_token_id = torch.argmax(next_token_logits).item()
        
        # 6. Append and Continue
        next_word = tokenizer.decode(next_token_id)
        print(next_word, end="", flush=True)
        
        generated_text += next_word
        
        # Stop if we generate the end-of-sentence token
        if next_token_id == tokenizer.eos_token_id:
            break

    print("\n\n✅ Generation Complete.")
    return generated_text

# --- Run the Test ---
# Try a simple prompt (LFM2 is an instruct model, so we can ask it questions)
final_output = generate_text("The capital of Poland is")

Loading Tokenizer...


[program.cpp:154] InternalConsistency verification requested but not available


Loading ExecuTorch Program: LFM2.5-1.2B-Instruct.pte...

🔵 Input: 'The capital of Poland is'
🟢 Generating:  Warsaw. It is the largest city in Poland and the country's political, economic, and cultural center

✅ Generation Complete.
